# Train PINN on miniature dataset

This notebook trains the **Physics-Informed Neural Network (PINN)** on the
miniature noiseless dataset.  The PINN augments a U-Net backbone with a
differentiable forward model that re-projects the predicted cube into
spectrogram space and compares it against the observed data.

## Loss function
$$\mathcal{L} = \underbrace{\lambda_{\text{rec}} \cdot \|\hat{y} - y\|^2}_{\text{reconstruction}} + \underbrace{\lambda_{\text{phys}} \cdot \|A(\hat{y}) - x_{\text{obs}}\|^2}_{\text{physics residual}}$$

where $A(\cdot)$ is the differentiable forward model (dispersion + PSF convolution)
implemented entirely in PyTorch so gradients flow through to the network weights.

## Why is the physics loss useful?
- At **inference time** (no ground-truth $y$ available), the physics loss still
  provides a meaningful gradient signal: it forces the predicted cube to be
  *consistent* with the observations.
- It acts as a strong physical regulariser that discourages hallucinated spectra
  that cannot explain the observed data.

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt

from spectangle.data_loaders.dataset import SpectangleDataModule
from spectangle.models.pinn import PINN
from spectangle.models.losses import CombinedLoss
from spectangle.utils.metrics import cube_metrics

# ── Config ─────────────────────────────────────────────────────────────────
H5_PATH        = Path('data/raw/simple_mini_100s.h5')
IN_CHANNELS    = 4
N_LAMBDA       = 128
IMAGE_SHAPE    = (128, 128)
LAMBDA_PHYSICS = 0.5          # weight of the physics loss
LR             = 5e-5
N_EPOCHS       = 5            # increase for real training
BATCH_SIZE     = 4
DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Using device: {DEVICE}')

## 1. Data loading

In [ ]:
dm = SpectangleDataModule(
    h5_path=H5_PATH,
    batch_size=BATCH_SIZE,
    n_channels=IN_CHANNELS,
    normalise='per_sample',
    num_workers=0,
    seed=42,
)
print(dm)

scene_shape       = dm.scene_shape
spectrogram_shape = dm.spectrogram_shape
print(f'Scene shape      : {scene_shape}')
print(f'Spectrogram shape: {spectrogram_shape}')
print(f'pad_y={dm.pad_y}, pad_x={dm.pad_x}')

## 2. Build PINN model
The PINN wraps a U-Net backbone with the differentiable forward model.
The `image_shape` must match the unpadded scene (cube) dimensions.

In [ ]:
pinn = PINN(
    in_channels=IN_CHANNELS,
    n_lambda=N_LAMBDA,
    image_shape=scene_shape,
    lambda_physics=LAMBDA_PHYSICS,
    psf_fwhm_pixels=1.6,
).to(DEVICE)

print(f'Parameters: {pinn.parameter_count():,}')
print(f'Physics model spectrogram shape: {pinn.physics_model.spectrogram_shape}')

# Sanity check: forward pass
x_dummy = torch.zeros(2, IN_CHANNELS, *spectrogram_shape, device=DEVICE)
out_dummy = pinn(x_dummy)
print(f'Output shape: {out_dummy.shape}  (expected: [2, {N_LAMBDA}, {scene_shape[0]}, {scene_shape[1]}])')

## 3. Training with physics loss
We use `forward_with_physics_loss` which returns the predicted cube plus both
the reconstruction and physics loss terms.  Monitoring these separately helps
diagnose whether the physics constraint is actually being satisfied.

In [ ]:
optimizer = torch.optim.AdamW(pinn.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

train_loader = dm.train_dataloader()
val_loader   = dm.val_dataloader()

history = {'train_total': [], 'train_rec': [], 'train_phys': [],
           'val_total':   [], 'val_rec':   [], 'val_phys':   []}

for epoch in range(1, N_EPOCHS + 1):
    # ── Train ───────────────────────────────────────────────────────────
    pinn.train()
    t_tot = t_rec = t_phys = 0.0; n = 0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        cube_pred, phys_loss, rec_loss = pinn.forward_with_physics_loss(x, y)
        loss = rec_loss + LAMBDA_PHYSICS * phys_loss
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(pinn.parameters(), 1.0)
        optimizer.step()
        bs = x.shape[0]
        t_tot  += loss.item()      * bs
        t_rec  += rec_loss.item()  * bs
        t_phys += phys_loss.item() * bs
        n      += bs
    history['train_total'].append(t_tot / n)
    history['train_rec'].append(t_rec / n)
    history['train_phys'].append(t_phys / n)

    # ── Validate ─────────────────────────────────────────────────────────
    pinn.eval()
    v_tot = v_rec = v_phys = 0.0; n_v = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            _, phys_loss, rec_loss = pinn.forward_with_physics_loss(x, y)
            loss = rec_loss + LAMBDA_PHYSICS * phys_loss
            bs = x.shape[0]
            v_tot  += loss.item()      * bs
            v_rec  += rec_loss.item()  * bs
            v_phys += phys_loss.item() * bs
            n_v    += bs
    history['val_total'].append(v_tot / n_v)
    history['val_rec'].append(v_rec / n_v)
    history['val_phys'].append(v_phys / n_v)

    scheduler.step()
    lr_now = optimizer.param_groups[0]['lr']
    print(f'Epoch {epoch:3d}/{N_EPOCHS} | '
          f'train total={history["train_total"][-1]:.5f} '
          f'rec={history["train_rec"][-1]:.5f} '
          f'phys={history["train_phys"][-1]:.5f} | '
          f'val total={history["val_total"][-1]:.5f} | '
          f'lr={lr_now:.2e}')

# Plot loss curves
fig, axs = plt.subplots(1, 3, figsize=(14, 3))
for ax, key, title in [
    (axs[0], 'total', 'Total loss'),
    (axs[1], 'rec',   'Reconstruction loss'),
    (axs[2], 'phys',  'Physics residual'),
]:
    ax.plot(history[f'train_{key}'], label='train')
    ax.plot(history[f'val_{key}'],   label='val', ls='--')
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(True)
plt.suptitle('PINN training curves (miniature)', y=1.02)
plt.tight_layout(); plt.show()

## 4. Physics residual visualisation
For a validation sample, predict the cube, re-project via the differentiable
forward model, and compare to the observed spectrograms.  Large residuals
indicate regions where the predicted cube is *inconsistent* with the data.

In [ ]:
pinn.eval()
x_s, y_s = dm._val_ds[0]
x_in = x_s.unsqueeze(0).to(DEVICE)
y_in = y_s.unsqueeze(0).to(DEVICE)

with torch.no_grad():
    cube_pred = pinn(x_in)                              # (1, n_lambda, ny, nx)
    reproj    = pinn.physics_model(cube_pred)            # (1, 4, H_spec, W_spec)

obs  = x_in[0, :4].cpu().numpy()
rep  = reproj[0].cpu().numpy()
res  = rep - obs
pred = cube_pred[0].cpu().numpy()
gt   = y_s.numpy()

# Physics residuals per K-sequence channel
fig, axs = plt.subplots(3, 4, figsize=(14, 8))
for k in range(4):
    v = np.max(np.abs(res[k]))
    axs[0, k].imshow(obs[k], origin='lower', cmap='inferno')
    axs[0, k].set_title(f'Observed K{k+1}'); axs[0, k].axis('off')
    axs[1, k].imshow(rep[k], origin='lower', cmap='inferno')
    axs[1, k].set_title(f'Reprojected K{k+1}'); axs[1, k].axis('off')
    im = axs[2, k].imshow(res[k], origin='lower', cmap='bwr', vmin=-v, vmax=v)
    axs[2, k].set_title(f'Residual K{k+1}'); axs[2, k].axis('off')
    fig.colorbar(im, ax=axs[2, k], fraction=0.03)
plt.suptitle('Physics residuals: A(ŷ) − x_obs  (row 3)')
plt.tight_layout(); plt.show()

# Cube reconstruction quality
metrics = cube_metrics(pred, gt)
print('\nReconstruction metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')

# Broadband comparison
fig, axs = plt.subplots(1, 3, figsize=(12, 4))
axs[0].imshow(gt.sum(0),   cmap='gray',  origin='lower'); axs[0].set_title('GT broadband')
axs[1].imshow(pred.sum(0), cmap='gray',  origin='lower'); axs[1].set_title('PINN prediction')
axs[2].imshow(np.abs(pred.sum(0) - gt.sum(0)), cmap='magma', origin='lower'); axs[2].set_title('|Residual|')
for a in axs: a.axis('off')
plt.tight_layout(); plt.show()

## 5. Effect of λ_physics
Run a quick ablation by setting `LAMBDA_PHYSICS=0` and comparing to the result
above to see how the physics constraint affects reconstruction quality.

In [ ]:
# Build a second PINN with physics loss disabled
pinn_noreg = PINN(
    in_channels=IN_CHANNELS,
    n_lambda=N_LAMBDA,
    image_shape=scene_shape,
    lambda_physics=0.0,   # no physics loss
).to(DEVICE)

opt2 = torch.optim.AdamW(pinn_noreg.parameters(), lr=LR, weight_decay=1e-5)
for epoch in range(N_EPOCHS):
    pinn_noreg.train()
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        _, _, rec_loss = pinn_noreg.forward_with_physics_loss(x, y)
        opt2.zero_grad(); rec_loss.backward(); opt2.step()

pinn_noreg.eval()
with torch.no_grad():
    pred_noreg = pinn_noreg(x_s.unsqueeze(0).to(DEVICE)).squeeze(0).cpu().numpy()

m_noreg = cube_metrics(pred_noreg, gt)
m_pinn  = cube_metrics(pred,       gt)

print('Metrics comparison (val sample 0):')
print(f'{"Metric":10s}  {"No physics":>12s}  {"PINN":>12s}')
for k in ['psnr', 'ssim', 'sam', 'rmse']:
    print(f'{k:10s}  {m_noreg[k]:>12.4f}  {m_pinn[k]:>12.4f}')

## Next steps
- Run full training via: `python scripts/train.py --config configs/models/pinn.yaml`
- Try `lambda_physics` values in [0.1, 0.5, 1.0, 5.0] and plot a λ sweep.
- Move to the euclid-like PINN notebook for noisy multi-order data.